# Kittra – Phase 2 training on Colab / Kaggle

Trains **ACT** and **Diffusion Policy** (LeRobot) on Kittra datasets and evaluates them in simulation.

1. Runtime → Change runtime type → **GPU** (T4 is enough for ACT; Diffusion is slower).
2. Run the cells top to bottom. Each training run saves checkpoints under `outputs/`.
3. Copy `outputs/` and `results/` back to your laptop (Drive or the Hub) at the end.

Datasets can be recorded here (cell 4a) or recorded on your laptop and pushed to the Hub (cell 4b).

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"          # headless rendering on the GPU
REPO = "https://github.com/keyurck7/KITTRA.git"
!git clone -q $REPO /content/kittra || (cd /content/kittra && git pull -q)
%cd /content/kittra
!pip install -q -r requirements-train.txt && pip install -q -e .

In [ ]:
# Sanity check: the scripted expert must succeed before recording data.
!python scripts/run_expert.py --episodes 10 --config randomized

### 4a. Record datasets here
200 successful episodes per variant. The 50/100-demo variants are exact prefixes (same seeds).

In [ ]:
!python scripts/record_lerobot_dataset.py --config nominal    --episodes 200 --overwrite
!python scripts/record_lerobot_dataset.py --config randomized --episodes 200 --overwrite
!python scripts/record_lerobot_dataset.py --config randomized --episodes 50  --overwrite
!python scripts/record_lerobot_dataset.py --config randomized --episodes 100 --overwrite

### 4b. …or use datasets you pushed from your laptop
Run `huggingface-cli login`, then pass `--hf-user <you>` to `train_lerobot.py` instead of recording.

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
HF_USER = None   # e.g. "keyurck7" when the datasets are on the Hub
hub = f"--hf-user {HF_USER}" if HF_USER else ""

### 5. Train
Full matrix = {ACT, Diffusion} × {nominal, randomized} × {50, 100, 200 demos}. Start with the two 200-demo randomized runs.
Reduce `--steps` for a quick smoke test.

In [ ]:
!python scripts/train_lerobot.py --config configs/train/act.yaml       --dataset kittra_randomized $hub --device cuda
!python scripts/train_lerobot.py --config configs/train/diffusion.yaml --dataset kittra_randomized $hub --device cuda

In [ ]:
# Remaining runs of the matrix (run when you have GPU time left)
for ds in ["kittra_nominal", "kittra_randomized_50", "kittra_randomized_100"]:
    for cfg in ["act", "diffusion"]:
        !python scripts/train_lerobot.py --config configs/train/{cfg}.yaml --dataset {ds} $hub --device cuda

### 6. Evaluate (fixed seeds, 100 episodes per condition)

In [ ]:
runs = {
    "act_randomized_200":       ("act",       "randomized", 200),
    "diffusion_randomized_200": ("diffusion", "randomized", 200),
    "act_nominal_200":          ("act",       "nominal",    200),
    "diffusion_nominal_200":    ("diffusion", "nominal",    200),
}
for name, (ptype, data, n) in runs.items():
    ckpt = f"outputs/{ptype}_{data}/checkpoints/last/pretrained_model"
    if os.path.exists(ckpt):
        !python scripts/evaluate.py --policy-type {ptype} --policy {ckpt} --name {name} --train-data {data} --n-demos {n} --conditions nominal randomized ood --episodes 100 --device cuda
!python scripts/make_results_table.py --runs results/runs --out results/phase2_table.md --plots results/plots

### 7. Save everything

In [ ]:
!zip -qr kittra_outputs.zip outputs results -x "*/optimizer_state*"
from google.colab import files
files.download("kittra_outputs.zip")